In [34]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [35]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Add a 20km buffer to the WDPA feature collection

In [36]:
features = ee.FeatureCollection("projects/deforestation-495419/assets/panama_protected_areas_polygons").filterBounds(panama_geom)

# Define the buffer function (Distance is in meters: 20km = 20000m) 
def add_buffer(feature):
    return feature.buffer(20000)

# Map the function over the FeatureCollection
buffered_features = features.map(add_buffer)

In [37]:
import pandas as pd
import ee

ee.Initialize()

# Geometries and Datasets
panama_geom = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

provinces = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

### Landuse aggregation and class mapping

In [38]:
# Reclassified Land Cover Image
sinia_prism = ee.Image("projects/deforestation-panama/assets/landuse_2021").clip(panama_geom)

lu_index = [10,20,30,40, 50, 60, 70, 80]

VisParams = {
    "min": 10,
    "max": 80, 
    "palette": [
        '32a65e',  # [10] Mature forest
        '32a65e',  # [20] Secondary forest
        '1f8d49',  # [30] Primary forest
        '7dc975',  # [40] Crops
        '04381d',  # [50] Plantation
        '026975',  # [60] Other vegetation
        '000000',  # [70] Other landuse
        '7a6c00',  # [80] Other forest 
       ]
    }
        
    # A flat list of pixel values to replace
from_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
    # A corresponding list of new values
to_list = [10, 20, 80, 80, 80, 80, 20, 20, 60, 60, 60, 70, 70, 50, 40, 50, 40, 40, 40, 50, 40, 40, 40, 40, 50, 60, 27, 28, 29, 70, 70, 70, 70]

    # for each forest age, mask the lulc of the year immediately preceding abandonment
lulc_aggregated = ee.Image()
remapped_band = sinia_prism.remap(from_list, to_list)

# Remapped image with single band
lulc_aggregated = sinia_prism.remap(from_list, to_list).rename('landuse')

# Map pixel values to string labels for readable DataFrame outputs
class_labels = {
    '10': 'Mature forest',
    '20': 'Secondary forest',
    '30': 'Primary forest',
    '40': 'Crops',
    '50': 'Plantation',
    '60': 'Other vegetation',
    '70': 'Other landuse',
    '80': 'Other forest',
    '27': 'Water surface',
    '28': 'Populated area',
    '29': 'Infrastructure'
}

### Climate Image Stacks (2014–2024)

In [39]:
years = list(range(2014, 2025))        # Full window for TerraClimate & CHIRPS
aeti_years = list(range(2018, 2025))   # WaPOR v3 availability starts in 2018

# Avg annual temp
terraclim = (
    ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
    .filterDate("2014-01-01", "2024-12-31")
    .filterBounds(panama_geom)
    .select(["tmmx", "tmmn"])
)

def calculate_yearly_temp(year):
    year_data = terraclim.filter(ee.Filter.calendarRange(year, year, "year")).map(lambda img: img.multiply(0.1))
    maxtemp = year_data.select("tmmx").reduce(ee.Reducer.mean())
    mintemp = year_data.select("tmmn").reduce(ee.Reducer.mean())
    return maxtemp.addBands(mintemp).reduce(ee.Reducer.mean()).float().rename(f"temp_{year}")

yearly_terraclim = ee.Image.cat([calculate_yearly_temp(yr) for yr in years])

# --- CHIRPS (Precipitation) ---
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")

def calculate_yearly_precip(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    return chirps.filterDate(start, end).sum().float().rename(f"precip_{year}")

yearly_chirps = ee.Image.cat([calculate_yearly_precip(yr) for yr in years])

### Avg annual evapotranspiration as a proxy for biomass

In [40]:
wapor_aeti = (
    ee.ImageCollection("FAO/WAPOR/3/L1_AETI_D")
    .filterDate("2018-01-01", "2024-12-31")
    .filterBounds(panama_geom)
    .select("L1-AETI-D")  # Fix 1: Correct band name
)

def calculate_yearly_aeti(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")
    # Multiply by 0.1 scale factor to get total mm/year
    annual_sum = wapor_aeti.filterDate(start, end).sum().multiply(0.1).float().rename(f"aeti_{year}")
    return annual_sum

yearly_aeti = ee.Image.cat([calculate_yearly_aeti(yr) for yr in aeti_years])

In [41]:
def process_protected_area(pa):
    pa_geom = pa.geometry()
    
    # Surface Area (km²)
    total_area_km2 = pa_geom.area().divide(1e6)
    
    # --- Province Overlap Analysis ---
    intersecting_provinces = provinces.filterBounds(pa_geom)
    
    def get_overlap(prov):
        prov_geom = prov.geometry()
        overlap_area = pa_geom.intersection(prov_geom, ee.ErrorMargin(1)).area().divide(1e6)
        return ee.Feature(None, {
            'prov_name': prov.get('ADM1_NAME'),
            'overlap_area': overlap_area
        })
    
    sorted_overlaps = intersecting_provinces.map(get_overlap).sort('overlap_area', False)
    prov_names = sorted_overlaps.aggregate_array('prov_name')
    
    main_prov = ee.Algorithms.If(
        prov_names.size().gt(0),
        prov_names.get(0),
        "Marine / Offshore"
    )
    
    other_provs = ee.Algorithms.If(
        prov_names.size().gt(1),
        prov_names.slice(1).join(", "),
        "None"
    )
    
    # --- Reductions ---
    histogram = lulc_aggregated.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=pa_geom,
        scale=30,
        maxPixels=1e9
    ).get('landuse')
    
    mean_temps = yearly_terraclim.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pa_geom,
        scale=4638,
        maxPixels=1e9
    )
    
    mean_precip = yearly_chirps.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pa_geom,
        scale=5566,
        maxPixels=1e9
    )
    
    mean_aeti = yearly_aeti.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pa_geom,
        scale=250,
        maxPixels=1e9
    )
    
    props = pa.toDictionary()
    
    out_dict = ee.Dictionary({
        'NAME': props.get('NAME', 'N/A'),
        'MAIN_PROVINCE': main_prov,
        'OTHER_PROVINCES': other_provs,
        'GIS_AREA_KM2': total_area_km2,
        'REP_AREA_KM2': props.get('REP_AREA', 'N/A'),
        'HISTOGRAM': histogram
    })
    
    combined = out_dict.combine(mean_temps).combine(mean_precip).combine(mean_aeti)
    return ee.Feature(None, combined)

### Processing and dataframe creation

In [42]:
processed_pas = panama_pas.map(process_protected_area)
features_info = processed_pas.getInfo()['features']

temp_cols = [f"temp_{yr}" for yr in years]
precip_cols = [f"precip_{yr}" for yr in years]
aeti_cols = [f"aeti_{yr}" for yr in aeti_years]  # Fix 2: 2018–2024 columns only

rows = []
for f in features_info:
    props = f['properties']
    counts = props.get('HISTOGRAM', {})
    
    if counts:
        sorted_classes = sorted(counts.items(), key=lambda item: item[1], reverse=True)
        dominant_code = sorted_classes[0][0]
        dominant_label = class_labels.get(dominant_code, f"Class {dominant_code}")
        
        other_labels = [
            class_labels.get(code, f"Class {code}") 
            for code, count in sorted_classes[1:]
        ]
        other_landuse_str = ", ".join(other_labels) if other_labels else "None"
    else:
        dominant_label = "No Data / Marine"
        other_landuse_str = "None"
    
    row = {
        'NAME': props.get('NAME'),
        'MAIN_PROVINCE': props.get('MAIN_PROVINCE'),
        'OTHER_PROVINCES': props.get('OTHER_PROVINCES'),
        'GIS_AREA_KM2': props.get('GIS_AREA_KM2'),
        'REP_AREA_KM2': props.get('REP_AREA_KM2'),
        'DOMINANT_LANDUSE': dominant_label,
        'OTHER_LANDUSES': other_landuse_str
    }
    
    # Temperature (°C)
    for col in temp_cols:
        val = props.get(col)
        row[col] = round(val, 2) if val is not None else None

    # Precipitation (mm/yr)
    for col in precip_cols:
        val = props.get(col)
        row[col] = round(val, 1) if val is not None else None

    # AETI (mm/yr)
    aeti_vals = []
    for col in aeti_cols:
        val = props.get(col)
        rounded_val = round(val, 1) if val is not None else None
        row[col] = rounded_val
        if rounded_val is not None:
            aeti_vals.append(rounded_val)

    # Compute AETI Summary Metrics
    if aeti_vals:
        row['AVG_ANNUAL_AETI_MM'] = round(sum(aeti_vals) / len(aeti_vals), 1)
        
        aeti_2024 = row.get('aeti_2024')
        aeti_2023 = row.get('aeti_2023')
        
        if aeti_2024 is not None and aeti_2023 is not None:
            diff = aeti_2024 - aeti_2023
            if diff > 0:
                row['AETI_STATUS'] = f"Improving (+{round(diff, 1)} mm)"
            elif diff < 0:
                row['AETI_STATUS'] = f"Declining ({round(diff, 1)} mm)"
            else:
                row['AETI_STATUS'] = "Stable"
        else:
            row['AETI_STATUS'] = "Insufficient Data"
    else:
        row['AVG_ANNUAL_AETI_MM'] = None
        row['AETI_STATUS'] = "No Data / Marine"

    rows.append(row)

df = pd.DataFrame(rows)

df['GIS_AREA_KM2'] = df['GIS_AREA_KM2'].round(2)

columns_order = [
    'NAME', 
    'MAIN_PROVINCE', 
    'OTHER_PROVINCES', 
    'GIS_AREA_KM2', 
    'REP_AREA_KM2', 
    'DOMINANT_LANDUSE', 
    'OTHER_LANDUSES',
    'AVG_ANNUAL_AETI_MM',
    'AETI_STATUS'
] + temp_cols + precip_cols + aeti_cols

df = df[columns_order]
df = df.sort_values(by='GIS_AREA_KM2', ascending=False).reset_index(drop=True)

print(df.head)

<bound method NDFrame.head of                          NAME      MAIN_PROVINCE    OTHER_PROVINCES  \
0         Cordillera de Coiba  Marine / Offshore               None   
1                Banco Volcán  Marine / Offshore               None   
2                      Darién             Darién             Emberá   
3                      Darién             Darién             Emberá   
4     Parc national du Darien             Darién  Emberá, Kuna Yala   
..                        ...                ...                ...   
73                Isla Iguana         Los Santos               None   
74  Manglares de Panamá Viejo             Panamá               None   
75        Punta Bruja y Dejal             Panamá               None   
76     El Salto de Las Palmas           Veraguas               None   
77   Zona de Reserva Matumbal     Bocas del Toro               None   

    GIS_AREA_KM2  REP_AREA_KM2  DOMINANT_LANDUSE  \
0       68204.74  67908.978834  No Data / Marine   
1       14269

In [43]:
df

,NAME,MAIN_PROVINCE,OTHER_PROVINCES,GIS_AREA_KM2,REP_AREA_KM2,DOMINANT_LANDUSE,OTHER_LANDUSES,AVG_ANNUAL_AETI_MM,AETI_STATUS,temp_2014,...,precip_2022,precip_2023,precip_2024,aeti_2018,aeti_2019,aeti_2020,aeti_2021,aeti_2022,aeti_2023,aeti_2024
0,Cordillera de Coiba,Marine / Offshore,None,68204.74,67908.978834,No Data / Marine,None,NaN,No Data / Marine,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Banco Volcán,Marine / Offshore,None,14269.95,14201.134174,No Data / Marine,None,NaN,No Data / Marine,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Darién,Darién,Emberá,5699.46,5689.578508,Mature forest,"Secondary forest, Other vegetation, Plantation...",104.4,Declining (-0.2 mm),25.01,...,3870.9,2752.5,3422.2,103.7,107.1,101.0,101.7,102.2,107.7,107.5
3,Darién,Darién,Emberá,5699.46,5689.578508,Mature forest,"Secondary forest, Other vegetation, Plantation...",104.4,Declining (-0.2 mm),25.01,...,3870.9,2752.5,3422.2,103.7,107.1,101.0,101.7,102.2,107.7,107.5
4,Parc national du Darien,Darién,"Emberá, Kuna Yala",5490.05,5790.000000,Mature forest,"Secondary forest, Other vegetation, Plantation...",104.7,Stable,25.03,...,3845.1,2735.4,3397.3,104.0,107.4,101.3,102.0,102.6,107.8,107.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,Isla Iguana,Los Santos,None,1.49,1.482343,Other vegetation,"Secondary forest, Other landuse",NaN,No Data / Marine,27.88,...,1135.3,1011.4,1416.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,Manglares de Panamá Viejo,Panamá,None,0.84,0.839527,Populated area,"Other forest, Water surface",150.7,Declining (-0.5 mm),27.91,...,2038.8,1399.7,1934.9,153.3,161.4,149.7,145.1,145.6,150.2,149.7
75,Punta Bruja y Dejal,Panamá,None,0.75,0.748226,Mature forest,"Other forest, Populated area",83.4,Improving (+2.4 mm),27.52,...,1942.8,1433.3,1896.6,82.0,77.2,86.4,77.0,83.9,87.5,89.9
76,El Salto de Las Palmas,Veraguas,None,0.56,0.558028,Other vegetation,"Secondary forest, Infrastructure",92.3,Improving (+10.4 mm),26.75,...,3948.2,2565.5,4308.7,93.8,93.4,93.8,100.3,89.1,82.7,93.1
